# Prompt SCI open-source LLMs through an API
In this notebook, you will get started with prompting through an API and using templates (parameterized) prompts. You will prompt open-source LLMs set up for use by Pitt's School of Computing and Information. 

We will use the `openai` Python package to make API calls to them.

In [1]:
import openai

# Create client and set API key
<span style="color:red">Fill in</span> the API key for the Pitt SCI open-source LLMs. You can get this from the class Canvas announcement.

In [2]:
class_sci_api_key = 'sk-g6Y1MHJaD_y1bohW7m_ItA'
client = openai.OpenAI( 
    api_key=class_sci_api_key, 
    base_url="https://ol.sci.pitt.edu" # LiteLLM Proxy is OpenAI compatible, Read More: https://docs.litellm.ai/docs/proxy/user_keys 
) 

# Start prompting!
SCI Tech has set up 3 open-source models on their servers available for us to use:
* `gemma3` (gemma3:27b)
* `llama3.1` (llama3.1:70b)
* `deepseek-r1` (deepseek-r1:70b)

Feel free to try any of them.

A couple of example prompts are given below, but try your own, too!

In [3]:
import pandas as pd
import glob
def call_llm(prompt, model='deepseek-r1'):
    """ Make an API call to Pitt SCI LLM 
        Args:
            model: {gemma3, llama3.1, deepseek-r1}
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": prompt
        }]
    )
    return response.choices[0].message.content

df = pd.read_csv('testSubs.csv')

# Load already completed movie_ids
existing_files = glob.glob('summaries_batch_*.csv')
if existing_files:
    completed_df = pd.concat([pd.read_csv(f) for f in existing_files])
    completed_ids = set(completed_df['movie_id'].tolist())
    print(f"Resuming — {len(completed_ids)} movies already done")
else:
    completed_ids = set()
    print("Starting fresh")

batch = []
batch_num = len(existing_files) + 1  # start from next batch number

for _, row in df.iterrows():
    if row['movie_id'] in completed_ids:
        print(f"Skipping {row['movie_id']} (already done)")
        continue

    prompt = f"""Given these subtitles for a movie, please generate a summary of the plot.

SUBTITLES:
{row['subtitles']}
"""

    generated_summary = call_llm(prompt, model='deepseek-r1')

    batch.append({
        'movie_id': row['movie_id'],
        'generated_summary': generated_summary
    })

    print(f"Done: {row['movie_id']}")

    # Save every 10 and reset batch
    if len(batch) == 10:
        pd.DataFrame(batch).to_csv(f'summaries_batch_{batch_num}.csv', index=False)
        print(f"Saved batch {batch_num}")
        batch_num += 1
        batch = []  # reset so each file only has 10 rows

# Save any remaining
if batch:
    pd.DataFrame(batch).to_csv(f'summaries_batch_{batch_num}.csv', index=False)
    print("Saved final batch")

print("All done!")

Resuming — 397 movies already done
Skipping 9249578 (already done)
Skipping 9226114 (already done)
Skipping 9395989 (already done)
Skipping 9447919 (already done)
Skipping 9224567 (already done)
Skipping 9242316 (already done)
Skipping 9500071 (already done)
Skipping 9511814 (already done)
Skipping 9507853 (already done)
Skipping 9323016 (already done)
Skipping 9330933 (already done)
Skipping 9382574 (already done)
Skipping 9495049 (already done)
Skipping 9361417 (already done)
Skipping 9252286 (already done)
Skipping 9295541 (already done)
Skipping 9201013 (already done)
Skipping 9223070 (already done)
Skipping 9389082 (already done)
Skipping 9237964 (already done)
Skipping 9350953 (already done)
Skipping 9208124 (already done)
Skipping 9227393 (already done)
Skipping 9498452 (already done)
Skipping 9437146 (already done)
Skipping 9459510 (already done)
Skipping 9364707 (already done)
Skipping 9309522 (already done)
Skipping 9362189 (already done)
Skipping 9467968 (already done)
Skipp

In [4]:
import glob

all_files = glob.glob('summaries_batch_*.csv')
combined_df = pd.concat([pd.read_csv(f) for f in sorted(all_files)])
combined_df.to_csv('summaries_output.csv', index=False)
print("Combined!")

Combined!
